# Bidirectional LSTM

Sentiment Analysis

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense

beauty_reviews = [
    "This new serum gave me glowing skin in just one week, highly recommend!",
    "The mascara was clumpy and smudged easily, terrible product.",
    "Loved the lightweight foundation that matched my skin tone perfectly.",
    "This lipstick dried out my lips and the color faded within an hour.",
    "The shampoo left my hair shiny and voluminous, best purchase ever.",
    "Disappointed with the moisturizer as it caused irritation and redness.",
    "This eyeshadow palette has beautiful colors and lasts all day.",
    "The cleanser was too harsh and made my skin dry and flaky.",
    "Absolutely love this moisturizer, it feels so hydrating and lightweight.",
    "The foundation oxidized and became too dark on my skin."
]

labels = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])

VOCAB_SIZE = 300
MAX_LENGTH = 20

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(beauty_reviews)
sequences = tokenizer.texts_to_sequences(beauty_reviews)
padded_data = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post')

model = Sequential([
    Input(shape=(MAX_LENGTH,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=32),
    Bidirectional(LSTM(units=32)),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("--- Beauty Products Sentiment Analysis Model ---")
model.summary()

print("\nTraining the model...")
model.fit(padded_data, labels, epochs=30, batch_size=4, verbose=0)

print("\n=== Beauty Product Sentiment Analysis ===")
while True:
    test_review = input("\nEnter a beauty product review (or type 'exit' to quit): ")
    if test_review.lower() == 'exit':
        break
    if not test_review.strip():
        continue

    test_seq = tokenizer.texts_to_sequences([test_review])
    test_padded = pad_sequences(test_seq, maxlen=MAX_LENGTH, padding='post')

    prediction = model.predict(test_padded, verbose=0)[0][0]

    print(f"\nReview: '{test_review}'")
    print(f"Sentiment Score: {prediction:.4f}")
    print("✅ HIGHLY RECOMMENDED" if prediction > 0.5 else "❌ NOT RECOMMENDED")

--- Beauty Products Sentiment Analysis Model ---


Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_17 (Embedding)        │ (None, 20, 32)         │         9,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_9 (Bidirectional) │ (None, 64)             │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,305 (102.75 KB)

 Trainable params: 26,305 (102.75 KB)

 Non-trainable params: 0 (0.00 B)


Training the model...

=== Beauty Product Sentiment Analysis ===

Enter a beauty product review (or type 'exit' to quit): this mascara gives volume

Review: 'this mascara gives volume'
Sentiment Score: 0.9550
✅ HIGHLY RECOMMENDED

Enter a beauty product review (or type 'exit' to quit): exit


Text/Sentence Autocompletion

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from collections import defaultdict, Counter

beauty_reviews = [
    "This new serum gave me glowing skin in just one week, highly recommend!",
    "The mascara was clumpy and smudged easily, terrible product.",
    "Loved the lightweight foundation that matched my skin tone perfectly.",
    "This lipstick dried out my lips and the color faded within an hour.",
    "The shampoo left my hair shiny and voluminous, best purchase ever.",
    "Disappointed with the moisturizer as it caused irritation and redness.",
    "This eyeshadow palette has beautiful colors and lasts all day.",
    "The cleanser was too harsh and made my skin dry and flaky.",
    "Absolutely love this moisturizer, it feels so hydrating and lightweight.",
    "The foundation oxidized and became too dark on my skin."
]

VOCAB_SIZE = 300

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(beauty_reviews)
sequences = tokenizer.texts_to_sequences(beauty_reviews)

transitions = defaultdict(list)
for seq in sequences:
    for i in range(len(seq) - 1):
        transitions[seq[i]].append(seq[i + 1])

print("--- Beauty Products Sentence Autocompletion ---")

while True:
    partial = input("\nEnter partial sentence (or type 'exit' to quit): ")
    if partial.lower() == 'exit':
        break
    if not partial.strip():
        continue

    seq = tokenizer.texts_to_sequences([partial])[0]
    if not seq:
        print("No recognized words in your input.")
        continue

    last_token = seq[-1]
    candidates = transitions.get(last_token, [])

    if not candidates:
        print("No suggestions available.")
        continue

    count = Counter(candidates)
    top_suggestions = count.most_common(5)

    suggested_words = [tokenizer.index_word.get(idx, "<OOV>") for idx, _ in top_suggestions]

    print(f"\nPartial: '{partial}'")
    print("Suggested next words:")
    for i, word in enumerate(suggested_words, 1):
        print(f"  {i}. {word}")

    best_word = suggested_words[0]
    completed = partial + " " + best_word
    print(f"\nExample completed: '{completed}'")

--- Beauty Products Sentence Autocompletion ---

Enter partial sentence (or type 'exit' to quit): eyeshadow

Partial: 'eyeshadow'
Suggested next words:
  1. palette

Example completed: 'eyeshadow palette'

Enter partial sentence (or type 'exit' to quit): exit
